# Sortformer diarization (Google Colab, batch)

Diarizes several interviews with NVIDIA Sortformer (NeMo) and writes the same
JSON shape as `src/diarization/pyannote_runner.py`.

**Prepare locally:**

```powershell
.\.venv\Scripts\python.exe -m scripts.colab_batch pack-audio --response-ids <id> <id> ...
```

**Input (upload to `/content`):** `sortformer_batch.zip` from `data\private\colab\`,
holding `<response_id>.wav` files (16 kHz mono).

**Output (downloaded):** `sortformer_results.zip` with
`<response_id>_diarization_sortformer.json` per interview. Unpack locally:

```powershell
.\.venv\Scripts\python.exe -m scripts.colab_batch unpack-diarization <path to sortformer_results.zip>
```

**Privacy:** clear all outputs before saving this notebook into the repository.
Check the model card licence before any use beyond this experiment.


In [ ]:
!pip install -q "nemo_toolkit[asr]"

In [ ]:
from pathlib import Path

ZIP_PATH = Path("/content/sortformer_batch.zip")
EXTRACT_DIR = Path("/content/sortformer_batch")
OUTPUT_DIR = Path("/content/sortformer_results")
RESULTS_ZIP = Path("/content/sortformer_results.zip")

# Streaming Sortformer handles long recordings; v1 is the offline model.
MODEL_NAME = "nvidia/diar_streaming_sortformer_4spk-v2"

In [ ]:
import zipfile

import soundfile as sf
from google.colab import files

if not ZIP_PATH.exists():
    print(f"Upload {ZIP_PATH.name}")
    files.upload()

if not ZIP_PATH.exists():
    raise FileNotFoundError(f"{ZIP_PATH.name} was not uploaded.")

EXTRACT_DIR.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(ZIP_PATH) as archive:
    archive.extractall(EXTRACT_DIR)

wav_paths = sorted(EXTRACT_DIR.rglob("*.wav"))
if not wav_paths:
    raise FileNotFoundError("No .wav files in the zip.")

for path in wav_paths:
    info = sf.info(str(path))
    print(f"{path.stem}  {info.duration / 60:.1f} min  {info.samplerate} Hz  ch={info.channels}")
    if info.samplerate != 16000 or info.channels != 1:
        raise ValueError(f"{path.name}: expected 16 kHz mono WAV.")

print(f"Interviews: {len(wav_paths)}")

In [ ]:
import torch
import nemo
from nemo.collections.asr.models import SortformerEncLabelModel

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

if DEVICE == "cpu":
    print("WARNING: no GPU. Runtime -> Change runtime type -> T4 GPU.")

diar_model = SortformerEncLabelModel.from_pretrained(MODEL_NAME)
diar_model = diar_model.to(DEVICE)
diar_model.eval()

# Streaming settings from the model card (high-latency, most accurate).
modules = getattr(diar_model, "sortformer_modules", None)
streaming = {
    "chunk_len": 340,
    "chunk_right_context": 40,
    "fifo_len": 40,
    "spkcache_update_period": 300,
}
if modules is not None:
    for name, value in streaming.items():
        if hasattr(modules, name):
            setattr(modules, name, value)

print(f"Model: {MODEL_NAME}  NeMo: {nemo.__version__}  Device: {DEVICE}")

In [ ]:
import datetime
import json


def parse_segment(item):
    # NeMo returns "start end speaker_N" strings; accept tuples too.
    if isinstance(item, str):
        start, end, speaker = item.split()[:3]
    else:
        start, end, speaker = list(item)[:3]
    return float(start), float(end), str(speaker)


OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
failures = []

for path in wav_paths:
    response_id = path.stem
    try:
        outputs = diar_model.diarize(audio=[str(path)], batch_size=1)

        # One list of segments per input file; some versions return a tuple.
        if isinstance(outputs, tuple):
            outputs = outputs[0]
        raw_segments = outputs[0]

        segments = []
        for item in raw_segments:
            start, end, speaker = parse_segment(item)
            if end > start:
                segments.append({
                    "speaker_id": speaker,
                    "start_sec": round(start, 3),
                    "end_sec": round(end, 3),
                })
        segments.sort(key=lambda s: (s["start_sec"], s["end_sec"]))

        result = {
            "audio_id": response_id,
            "model_name": MODEL_NAME,
            "created_at_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(),
            "nemo_version": nemo.__version__,
            "segments": segments,
        }
        output_path = OUTPUT_DIR / f"{response_id}_diarization_sortformer.json"
        output_path.write_text(json.dumps(result, indent=2), encoding="utf-8")

        totals = {}
        for s in segments:
            totals[s["speaker_id"]] = totals.get(s["speaker_id"], 0) + s["end_sec"] - s["start_sec"]
        rounded = {k: round(v, 1) for k, v in sorted(totals.items())}
        print(f"{response_id}  segments={len(segments)}  totals(s)={rounded}")
    except Exception as error:
        failures.append(response_id)
        print(f"{response_id}  FAILED: {error!r}")

with zipfile.ZipFile(RESULTS_ZIP, "w") as archive:
    for output_path in sorted(OUTPUT_DIR.glob("*.json")):
        archive.write(output_path, output_path.name)

print()
print(f"Diarized: {len(wav_paths) - len(failures)} / {len(wav_paths)}  Failed: {failures}")
files.download(str(RESULTS_ZIP))